In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Simple CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3), nn.ReLU(),
            nn.Flatten()
        )
        self.fc = nn.Linear(64 * 11 * 11, 10)

    forward = lambda self, x: self.fc(self.conv(x))

In [ ]:
def client_update(client_model, optimizer, train_loader, epochs=1):
    """Train the local model on client data."""
    client_model.train()
    for _ in range(epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = client_model(data)
            loss = nn.CrossEntropyLoss()(output, target)
            loss.backward()
            optimizer.step()
    return client_model.state_dict(), len(train_loader.dataset)

def server_aggregate(global_model, client_weights, client_lens):
    """Perform Weighted Federated Averaging."""
    total_samples = sum(client_lens)
    global_dict = global_model.state_dict()

    for key in global_dict.keys():
        # Initialize the weight for this layer as zero
        weighted_sum = torch.zeros_like(global_dict[key], dtype=torch.float32)

        for i in range(len(client_weights)):
            # Apply weighting: (n_k / n) * weight_k
            contribution = client_weights[i][key].float() * (client_lens[i] / total_samples)
            weighted_sum += contribution

        global_dict[key].copy_(weighted_sum)

    global_model.load_state_dict(global_dict)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import copy

# --- Setup ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Prepare Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1000)

# 2. Distribute data unequally among 5 clients
num_clients = 5
indices = np.arange(len(train_dataset))
np.random.shuffle(indices)

# Creating unequal splits for 60,000 samples
# Client 0: 6000 | Client 1: 9000 | Client 2: 12000 | Client 3: 15000 | Client 4: 18000
split_points = [6000, 15000, 27000, 42000]
client_indices = np.split(indices, split_points)

# 3. Initialize Global Model
# (Assuming SimpleCNN is defined in your previous cells)
global_model = SimpleCNN().to(device)
rounds = 5

# 4. Federated Training Loop
for r in range(rounds):
    print(f"\n--- Communication Round {r+1} ---")
    client_weights = []
    client_lens = []

    # Distribution & Local Training phase
    for i in range(num_clients):
        # Server sends current global model to client
        local_model = copy.deepcopy(global_model)
        optimizer = optim.SGD(local_model.parameters(), lr=0.01)

        # Get client-specific data subset
        loader = DataLoader(Subset(train_dataset, client_indices[i]), batch_size=32, shuffle=True)

        # Local Update (Training)
        weights, n_k = client_update(local_model, optimizer, loader)
        client_weights.append(weights)
        client_lens.append(n_k)
        print(f"Client {i} trained on {n_k} samples.")

    # Server-Side Model Aggregation phase
    server_aggregate(global_model, client_weights, client_lens)

    # Global Evaluation phase
    global_model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = global_model(data)
            correct += output.argmax(dim=1).eq(target).sum().item()

    # Corrected Accuracy Calculation
    total_test_samples = len(test_loader.dataset)
    accuracy = 100. * correct / total_test_samples
    print(f"Global Accuracy: {accuracy:.2f}%")


--- Communication Round 1 ---
Client 0 trained on 6000 samples.
Client 1 trained on 9000 samples.
Client 2 trained on 12000 samples.
Client 3 trained on 15000 samples.
Client 4 trained on 18000 samples.
Global Accuracy: 93.07%

--- Communication Round 2 ---
Client 0 trained on 6000 samples.
Client 1 trained on 9000 samples.
Client 2 trained on 12000 samples.
Client 3 trained on 15000 samples.
Client 4 trained on 18000 samples.
Global Accuracy: 95.70%

--- Communication Round 3 ---
Client 0 trained on 6000 samples.
Client 1 trained on 9000 samples.
Client 2 trained on 12000 samples.
Client 3 trained on 15000 samples.
Client 4 trained on 18000 samples.
Global Accuracy: 96.67%

--- Communication Round 4 ---
Client 0 trained on 6000 samples.
Client 1 trained on 9000 samples.
Client 2 trained on 12000 samples.
Client 3 trained on 15000 samples.
Client 4 trained on 18000 samples.
Global Accuracy: 97.42%

--- Communication Round 5 ---
Client 0 trained on 6000 samples.
Client 1 trained on 900